In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Users/lhungen@gmail.com/FMCG_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://fmcg-databricks-bronze/{data_source}/*.csv'
print(base_path)


s3://fmcg-databricks-bronze/gross_price/*.csv


## Bronze

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
# print check data type
df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(df.limit(10))

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-03-27T00:21:50.153Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-03-27T00:21:50.153Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-03-27T00:21:50.153Z,gross_price.csv,2741


In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025/07/01|        -84|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|01/08/2025|    unknown|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025/09/01|         84|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-10-01|         83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-11-01|         83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  88888888|2025-12-01|        -83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-07-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-08-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-09-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  2

### Transformations

In [0]:
df_bronze.select('month').distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [0]:
## Normalized the month
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
df_silver.select('month').distinct().show()

#### Handle gross_price

In [0]:
df_silver.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025-07-01|        -84|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-08-01|    unknown|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-09-01|         84|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-10-01|         83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-11-01|         83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  88888888|2025-12-01|        -83|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-07-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-08-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-09-01|         68|2026-03-27 00:22:...|gross_price.csv|     2741|
|  2

In [0]:
# We are validating the gross_price column, converting only valid numeric values to double, fixing negative prices by making them positive, and replacing all non-numeric values with 0


df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
            .otherwise(F.col("gross_price").cast("double")))
    .otherwise(0)
)

In [0]:
df_silver.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025-07-01|       84.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-08-01|        0.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-09-01|       84.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-10-01|       83.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|2025-11-01|       83.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  88888888|2025-12-01|       83.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-07-01|       68.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-08-01|       68.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891102|2025-09-01|       68.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  2

In [0]:
# We enrich the silver dataset by performing an inner join with the products table to fetch the correct product_code for each product_id.

df_products = spark.table("fmcg.silver.products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")

df_joined.show(5)

+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|product_id|        product_code|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|  25891101|e91ba9d665f90254d...|2025-07-01|       84.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-08-01|        0.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-09-01|       84.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-10-01|       83.0|2026-03-27 00:22:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-11-01|       83.0|2026-03-27 00:22:...|gross_price.csv|     2741|
+----------+--------------------+----------+-----------+--------------------+---------------+---------+
only showing top 5 rows


In [0]:
df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true")\
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")
display(df_silver)

product_id,product_code,month,gross_price,read_timestamp,file_name,file_size
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-07-01,84.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-08-01,0.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-09-01,84.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-10-01,83.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891101,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,2025-11-01,83.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-07-01,68.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-08-01,68.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-09-01,68.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-10-01,69.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741
25891102,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,2025-11-01,69.0,2026-03-27T00:22:04.731Z,gross_price.csv,2741


In [0]:
# select only required columns
df_gold = df_silver.select("product_code", "month", "gross_price")
df_gold.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [0]:
df_gold.write\
  .format("delta")\
.option("delta.enableChangeDataFeed", "true")\
  .mode("overwrite")\
      .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merging Data source with parent

In [0]:
df_gold_price = spark.table("fmcg.gold.sb_dim_gross_price")
df_gold_price.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows
